# Trader Execution

Walk-forward is the test design for this chapter: train on the past, test on the next
unseen stretch, with an embargo gap so nothing leaks across. Two models are judged the
same way: the rule-based strategy (`inputs/walkforward.py`) and the machine-learning
classifier built here. Both are NO-GO so far; nothing trades.

This notebook runs the click-and-go ML workflow: 3A build the training and test data,
3B train and score the bake-off (logistic regression, random forest, LightGBM) with the
60/40 confidence filter, then 3C and 3D scaffolds. The reusable feature code lives in
`inputs/build_dataset.py` (shared with the live path so the training and live feature
vectors never drift); the modeling is inline below so you can edit it here.

### Environment

In [ ]:
import sys, subprocess
from pathlib import Path

REQUIRED_PY = (3, 11)
if sys.version_info[:2] != REQUIRED_PY:
    raise RuntimeError(
        f"This notebook needs Python {REQUIRED_PY[0]}.{REQUIRED_PY[1]}, "
        f"but the kernel is {sys.version.split()[0]}. Switch the kernel and re-run.")

REQ = Path("inputs/requirements.txt")
if REQ.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--break-system-packages", "--disable-pip-version-check",
                    "-r", str(REQ)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "--break-system-packages", "lightgbm"], check=False)  # Tier 1

import os, numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except ImportError:
    HAVE_LGBM = False

# Shared feature/data layer (single source of truth, also used by the live path).
for cand in ["inputs", os.path.join("..", "inputs")]:
    if os.path.isdir(cand):
        sys.path.insert(0, os.path.abspath(cand)); break
import build_dataset as bd

OUTPUTS = Path("outputs"); CSV = OUTPUTS / "CSV"; MODEL_DIR = OUTPUTS / "3B-model-training"
for d in (CSV, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"env ready  ·  python {sys.version.split()[0]}  ·  "
      f"lightgbm {'yes' if HAVE_LGBM else 'no'}  ·  {len(bd.FEATURES)} features")

## Training-Test Data

The labelled dataset is one row per coin per day: scale-invariant features plus a
forward label (1 if the close gains the target before the stop, within the horizon).
The split below trains on the older share and tests on the newer, with an embargo gap
of one label horizon on each side so a label cannot peek across the cut. A full rolling
walk-forward (retrain at each step) is the planned upgrade; this notebook does the
single embargoed split for now.

In [ ]:
CONFIG = dict(
    train_frac   = 0.70,          # older share trains, newer tests
    embargo_days = bd.HORIZON,    # gap each side of the cut = the label horizon
    conf_hi      = 0.60,          # confidence filter: act-long (Keller Metric 1)
    conf_lo      = 0.40,          # act-short / stand-aside
    gbm = dict(n_estimators=600, num_leaves=31, learning_rate=0.05, max_depth=6,
               min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
               subsample_freq=5),  # conservative gradient boosting (Keller)
)
print(f"label: +{bd.TARGET:.0%} before -{bd.STOP:.0%} within {bd.HORIZON}d  ·  "
      f"confidence band {CONFIG['conf_lo']}-{CONFIG['conf_hi']}")

### Import Data

Loads `outputs/CSV/dataset.csv` if present, otherwise builds it from live data (needs
network). The feature set is the 17 base features plus the Keller families added in
`build_dataset.py`; if the saved dataset predates them, the cell says how to rebuild.

In [ ]:
ds = CSV / "dataset.csv"
if ds.exists():
    df = pd.read_csv(ds, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    print(f"loaded {ds}  ({len(df):,} rows)")
else:
    print("no dataset.csv found - building from live data (needs network)...")
    df = bd.build(); df.to_csv(ds, index=False)

missing = [f for f in bd.FEATURES if f not in df.columns]
if missing:
    print(f"\nNOTE: the saved dataset is missing {len(missing)} of the new features "
          f"(e.g. {missing[:3]}).\nRebuild it to pick them up:  python inputs/build_dataset.py")
else:
    print(f"all {len(bd.FEATURES)} features present  ·  base rate {df['label'].mean():.3f}  ·  "
          f"{df['date'].min().date()} to {df['date'].max().date()}")

### Model Features Collection

The base stack measures trend, momentum, and shape. The Keller families add
multi-lookback momentum and its acceleration, realized-volatility estimators
(close-to-close, Parkinson, Garman-Klass), volatility-adjusted returns, and a
scale-invariant Amihud illiquidity. See `tasks/keller-integration.md`.

In [ ]:
base = bd.FEATURES[:17]
keller = bd.FEATURES[17:]
print(f"base features ({len(base)}):\n  " + ", ".join(base))
print(f"\nKeller families ({len(keller)}):\n  " + ", ".join(keller))
present = [f for f in bd.FEATURES if f in df.columns]
df[present].describe().T[["mean", "std", "min", "max"]].round(3)

## Model Training

Train on the training window, score once on the test window. Three models compete:
logistic regression, random forest, and LightGBM (gradient boosting, Tier 1). Each is
reported two ways: standard metrics at the 0.5 threshold, and the 60/40 confidence
filter (Keller Metric 1), which keeps only high-conviction rows. Read precision against
the base rate, which is well below 0.5.

In [ ]:
def split(df, frac, embargo_days):
    cut = df["date"].quantile(frac)
    emb = pd.Timedelta(days=embargo_days)
    return df[df["date"] <= cut - emb].copy(), df[df["date"] > cut + emb].copy(), cut

def confidence_filtered(y, prob, hi, lo):
    y = np.asarray(y); prob = np.asarray(prob)
    mask = (prob >= hi) | (prob <= lo); n = int(mask.sum())
    if n == 0:
        return dict(coverage=0.0, n=0, precision=np.nan, recall=np.nan, f1=np.nan)
    yt = y[mask]; yp = (prob[mask] >= 0.5).astype(int)
    return dict(coverage=float(mask.mean()), n=n,
                precision=precision_score(yt, yp, zero_division=0),
                recall=recall_score(yt, yp, zero_division=0),
                f1=f1_score(yt, yp, zero_division=0))

def evaluate(name, model, Xtr, ytr, Xte, yte, base):
    cv = TimeSeriesSplit(n_splits=5)
    try:
        cv_auc = cross_val_score(model, Xtr, ytr, cv=cv, scoring="roc_auc").mean()
    except Exception:
        cv_auc = float("nan")
    model.fit(Xtr, ytr)
    prob = model.predict_proba(Xte)[:, 1]; pred = (prob >= 0.5).astype(int)
    cf = confidence_filtered(yte, prob, CONFIG["conf_hi"], CONFIG["conf_lo"])
    print(f"\n--- {name} ---")
    print(f"  train CV AUC {cv_auc:.3f} | test AUC {roc_auc_score(yte, prob):.3f} | "
          f"acc {accuracy_score(yte, pred):.3f}")
    print(f"  precision(buy) {precision_score(yte, pred, zero_division=0):.3f}  "
          f"recall {recall_score(yte, pred, zero_division=0):.3f}  (base {base:.3f})")
    print(f"  60/40 filter keeps {cf['coverage']:.0%} (n={cf['n']}): "
          f"prec {cf['precision']:.3f} rec {cf['recall']:.3f} F1 {cf['f1']:.3f}")
    return dict(name=name, model=model, prob=prob, auc=roc_auc_score(yte, prob),
                prec=precision_score(yte, pred, zero_division=0))

In [ ]:
feat = [f for f in bd.FEATURES if f in df.columns]
tr, te, cut = split(df, CONFIG["train_frac"], CONFIG["embargo_days"])
Xtr, ytr, Xte, yte = tr[feat], tr["label"], te[feat], te["label"]
base_te = yte.mean()
print(f"train {len(tr):,}  test {len(te):,}  cut {pd.Timestamp(cut).date()}  "
      f"embargo +/-{CONFIG['embargo_days']}d  base rate {base_te:.3f}")

models = [
    ("LogisticRegression",
     Pipeline([("s", StandardScaler()),
               ("c", LogisticRegression(max_iter=2000, class_weight="balanced"))])),
    ("RandomForest",
     RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=50,
                            class_weight="balanced", n_jobs=-1, random_state=0)),
]
if HAVE_LGBM:
    models.append(("LightGBM",
                   LGBMClassifier(**CONFIG["gbm"], class_weight="balanced",
                                  random_state=0, n_jobs=-1, verbosity=-1)))

results = [evaluate(n, m, Xtr, ytr, Xte, yte, base_te) for n, m in models]
best = max(results, key=lambda r: r["prec"])
print(f"\nchosen: {best['name']}  (precision {best['prec']:.3f} vs base {base_te:.3f}, "
      f"AUC {best['auc']:.3f})")

In [ ]:
# Honesty gate and save. Precision must clearly beat the base rate and AUC clear 0.55.
go = (best["prec"] > base_te + 0.05) and (best["auc"] > 0.55)
print("HONESTY GATE:",
      "GO (edge survives out-of-sample)" if go
      else "NO-GO (no demonstrable edge - do not trade)")
joblib.dump({"model": best["model"], "features": feat, "name": best["name"],
             "trained_through": str(pd.Timestamp(cut).date()),
             "test_base_rate": float(base_te), "go": bool(go)},
            MODEL_DIR / "model.joblib")
print("saved", MODEL_DIR / "model.joblib")

In [ ]:
# Feature importance for the strongest tree model (what the model leans on).
tree = next((r for r in results if r["name"] in ("LightGBM", "RandomForest")), None)
if tree is not None:
    imp = getattr(tree["model"], "feature_importances_", None)
    if imp is not None:
        order = np.argsort(imp)[::-1][:15]
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh([feat[i] for i in order][::-1], imp[order][::-1], color="#0B3D66")
        ax.set_title(f"{tree['name']}: top 15 feature importances")
        fig.tight_layout()
        (OUTPUTS / "PNG").mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUTS / "PNG" / "3B-feature-importance.png", dpi=160)
        plt.show()

## Model Tuning

Search for settings that clear both baselines (buy-and-hold and a coin flip), every
candidate scored on the walk-forward test windows, after fees. Tune the confidence
cutoffs `conf_hi` and `conf_lo`, the vote threshold and entry selectivity, and the exit
geometry (`stop_atr_mult`, `take_profit_pct`, the diagnosed leak). Append one row per
run to `outputs/CSV/experiment_log.csv`.

## Stability

Confirm any edge is not an artifact: parameter stability, results split by market type,
coin-flip and buy-and-hold baselines, an optional bootstrap on trade returns. Only then
paper trade, then a tiny live allocation. No live trading until a configuration clearly
beats buy-and-hold and a coin flip, out-of-sample and after fees.